In [ ]:
import _plotting as plot
import jax
import jax.numpy as jnp
from matplotlib import pyplot as plt

from xxm.core.align import match_states_by_conditional_mean as match_states
from xxm.hmm import GaussianARHMM

In [ ]:
def make_true_model() -> GaussianARHMM:
    return GaussianARHMM.from_params(
        initial_probs=jnp.array([0.4, 0.3, 0.3]),
        transition_probs=jnp.array(
            [
                [0.97, 0.02, 0.01],
                [0.02, 0.97, 0.01],
                [0.02, 0.02, 0.96],
            ]
        ),
        emission_coefficients=jnp.array(
            [
                # clockwise rotation
                [
                    [[0.92, 0.32]],
                    [[-0.32, 0.92]],
                ],
                # counter-clockwise rotation
                [
                    [[0.92, -0.32]],
                    [[0.32, 0.92]],
                ],
                # anisotropic dynamics
                [
                    [[0.97, 0.00]],
                    [[0.00, 0.65]],
                ],
            ]
        ),  # (K=3, O=2, L=1, I=2)
        emission_bias=jnp.zeros((3, 2)),
        emission_covariances=jnp.array(
            [
                [[0.05, 0.00], [0.00, 0.05]],
                [[0.05, 0.00], [0.00, 0.05]],
                [[0.05, 0.00], [0.00, 0.05]],
            ]
        ),
    )


true_model = make_true_model()

true_states, observations = true_model.sample(
    num_steps=1000,
    key=jax.random.key(0),
)

In [ ]:
_, ax = plt.subplots()
plot.plot_seq_2d(ax, true_states, observations)

In [ ]:
plot.plot_dyn_conditional_linear(
    true_model.states,
)

In [ ]:
posterior, _ = true_model.infer(observations)

In [ ]:
observations

In [ ]:
plot.plot_seq_1d_comparison(
    true_states,
    observations,
    true_model.most_likely_states(posterior),
    true_model.observation_mean(observations, posterior),
)

In [ ]:
initial_model = GaussianARHMM.from_kmeans(
    key=jax.random.key(0),
    observations=observations,
    num_states=true_model.num_states,
    num_lags=1,
)


fit = initial_model.fit(
    observations=observations,
    num_iters=50,
)


plot.plot_fit_progress(fit.objective_trace, name='Log Likelihood')

In [ ]:
def align_model(learned_model, true_model):

    permutation = match_states(
        learned_model.states_conditional(observations).mean,
        true_model.states_conditional(observations).mean,
    )
    learned_model = learned_model.permute(permutation)

    return learned_model


learned_model = align_model(fit.model, true_model)

In [ ]:
plot.plot_dyn_conditional_linear_comparison(true_model.states, learned_model.states)

In [ ]:
posterior, _ = learned_model.infer(observations)

In [ ]:
plot.plot_seq_1d_comparison(
    true_states,
    observations,
    learned_model.most_likely_states(posterior),
    learned_model.observation_mean(observations, posterior),
)